In [3]:
# I didn't measure launch success so I will use (as close as I can)
# to the industry standard of 50% visibility score, 40% race performance,
# and 10% for special events (playoffs etc)

# since colinearity for finish position and laps led is incredibly high 
# I will just be using laps led, because it had stronger correlation with visibility 

# Define weight configuration
WEIGHT_CONFIG = {
    'category_weights': {
        'laps_led': 0.40,    # 40% of total score
        'visibility_score': 0.50,      # 50% of total score
        'special_events': 0.10       # 10% of total score
    },
    'variable_weights_within_category': {
        'laps_led': {
            'laps_led': 1.00
        },
        'special_events': {
            'is_win': 0.60,           # 60% of special events weight
            'is_playoff': 0.40        # 40% of special events weight
        },
        'visibility_score': {
            'visibility_score': 1.00
    }
}
}
# Calculate effective weights for each variable
def calculate_effective_weights(config):
    """
    Calculate the final weight for each variable.

    Parameters:
    -----------
    config : dict
        Weight configuration with category and variable weights

    Returns:
    --------
    dict : Effective weight for each variable
    """
    effective = {}
    for category, cat_weight in config['category_weights'].items():
        var_weights = config['variable_weights_within_category'].get(category, {})
        for var, var_weight in var_weights.items():
            effective[var] = cat_weight * var_weight
    return effective

effective_weights = calculate_effective_weights(WEIGHT_CONFIG)

print("Effective Variable Weights:")
print("="*50)
for var, weight in sorted(effective_weights.items(), key=lambda x: -x[1]):
    print(f"{var:<30}: {weight*100:>5.1f}%")
print(f"{'TOTAL':<30}: {sum(effective_weights.values())*100:>5.1f}%")

Effective Variable Weights:
visibility_score              :  50.0%
laps_led                      :  40.0%
is_win                        :   6.0%
is_playoff                    :   4.0%
TOTAL                         : 100.0%


In [5]:
weight_rationale = """
# Visibility Score: Weighting Rationale

## Overall Philosophy

The weighting scheme prioritizes **on-track performance and media exposure** because:

1. On-track results determine visibility opportunity (better finishes = more camera time)
2. Traditional media still reaches the largest audience for NASCAR content
3. Social engagement, while valuable, represents a subset of total audience

## Category Rationale

### Race Performance (40%)

*Why 40%?* Race performance is the foundation of sponsorship value. A sponsor on a winning car gets exponentially more exposure than one on a mid-pack car. Our EDA showed that finish position correlates strongly with all visibility metrics.

**Variable breakdown:**
- **Laps Led (100%)**: Laps led is a strong proxy for race dominance
### Visibility Score (50%)
*Why 50%?* Visibility score is a composite metric that captures the total exposure across all media channels. It is the most direct measure of how much attention a driver and their sponsors receive.
**Variable breakdown:**
- **Visibility Score (100%)**: This is the aggregated score from all visibility sources,
### Special Events (10%)

*Why 10%?* Wins and playoff races generate disproportionate visibility spikes that should be captured.

**Variable breakdown:**
- **Wins (60%)**: A win generates 3-5x normal visibility. Even a small bonus weight has large impact when activated.
- **Playoffs (40%)**: Playoff races have higher stakes and more viewers. Built-in visibility multiplier.

## Design Decisions

### Why not equal weights?

Equal weights (14.3% each for 7 variables) would:
- Treat Reddit mentions as equally important as race wins
- Ignore the hierarchical importance of performance vs. social

### Why not data-driven weights from regression?

Regression-based weights would:
- Require a labeled "visibility outcome" variable (which doesn't exist)
- Be sensitive to sample size and outliers
- Be harder to explain to stakeholders

The hybrid approach provides transparency and defensibility.

## Sensitivity Testing

We will test alternative weight configurations in Substep 4.3.3:
- Higher performance weight (50/25/15/10)
- Higher media weight (30/40/20/10)
- Equal weights (25/25/25/25)

If rankings change significantly under alternative weights, we'll note the sensitivity in our methodology.
"""

# Save rationale document
with open('weighting_rationale.md', 'w') as f:
    f.write(weight_rationale)

print("Weighting rationale saved to: weighting_rationale.md")

Weighting rationale saved to: weighting_rationale.md


In [8]:
import json

# Full configuration for scoring model
SCORING_CONFIG = {
    'version': '1.0',
    'created_date': '2026-07-10',  # Update with actual date
    'category_weights': {
        'race_performance': 0.40,
        'visibility_score': 0.50,
        'special_events': 0.10
    },
    'variables': {
        'finish_position': {
            'category': 'race_performance',
            'weight_within_category': 0,
            'effective_weight': 0,
            'direction': 'inverse',
            'transform': 'invert_position'
        },
        'laps_led': {
            'category': 'race_performance',
            'weight_within_category': 1,
            'effective_weight': 0.4,
            'direction': 'direct',
            'transform': 'normalize'
        },
        'news_weighted_mentions': {
            'category': 'visibility_score',
            'weight_within_category': 'N/A',  # Placeholder for actual weight
            'effective_weight': 'N/A',  # Placeholder for actual effective weight
            'direction': 'direct',
            'transform': 'normalize'
        },
        'reddit_mentions': {
            'category': 'visibility_score',
            'weight_within_category': 'N/A',  # Placeholder for actual weight
            'effective_weight': 'N/A',  # Placeholder for actual effective weight
            'direction': 'direct',
            'transform': 'normalize'
        },
        'youtube_sponsor_views': {
            'category': 'visibility_score',
            'weight_within_category': 'N/A',  # Placeholder for actual weight
            'effective_weight': 'N/A',  # Placeholder for actual effective weight
            'direction': 'direct',
            'transform': 'normalize'
        },
        'is_win': {
            'category': 'special_events',
            'weight_within_category': 0.60,
            'effective_weight': 0.06,
            'direction': 'direct',
            'transform': 'binary'
        },
        'is_playoff': {
            'category': 'special_events',
            'weight_within_category': 0.40,
            'effective_weight': 0.04,
            'direction': 'direct',
            'transform': 'binary'
        }
    }
}

# Save configuration
with open('scoring_config.json', 'w') as f:
    json.dump(SCORING_CONFIG, f, indent=2)

print("Scoring configuration saved to: scoring_config.json")

Scoring configuration saved to: scoring_config.json


In [10]:
import pandas as pd

def normalize_to_100(series, direction='direct'):
    """
    Normalize a series to 0-100 scale.

    Parameters:
    -----------
    series : pd.Series
        Values to normalize
    direction : str
        'direct' if higher is better, 'inverse' if lower is better

    Returns:
    --------
    pd.Series : Normalized values (0-100 scale)
    """
    min_val = series.min()
    max_val = series.max()

    if max_val == min_val:
        # Avoid division by zero
        return pd.Series([50] * len(series), index=series.index)

    if direction == 'direct':
        # Higher original = higher normalized (0 at min, 100 at max)
        normalized = (series - min_val) / (max_val - min_val) * 100
    elif direction == 'inverse':
        # Lower original = higher normalized (100 at min, 0 at max)
        normalized = (max_val - series) / (max_val - min_val) * 100
    else:
        raise ValueError(f"Direction must be 'direct' or 'inverse', got {direction}")

    return normalized

# Test on sample data
sample = pd.Series([1, 5, 10, 20, 40])  # Finish positions

print("Sample Finish Positions:", sample.tolist())
print("Normalized (inverse):", normalize_to_100(sample, 'inverse').round(1).tolist())

Sample Finish Positions: [1, 5, 10, 20, 40]
Normalized (inverse): [100.0, 89.7, 76.9, 51.3, 0.0]


In [11]:
def prepare_binary(series):
    """
    Convert binary variable to 0-100 scale.

    Parameters:
    -----------
    series : pd.Series
        Binary values (0 or 1)

    Returns:
    --------
    pd.Series : Scaled to 0-100 (0 stays 0, 1 becomes 100)
    """
    return series * 100

# Test
wins = pd.Series([0, 0, 1, 0, 0])
print("\\nWin Flag Scoring:")
print(pd.DataFrame({'is_win': wins, 'score': prepare_binary(wins)}))

\nWin Flag Scoring:
   is_win  score
0       0      0
1       0      0
2       1    100
3       0      0
4       0      0


In [14]:
AGGREGATION_METHODS = {
    'season_total': {
        'method': 'sum',
        'description': 'Sum of all weekly visibility scores',
        'use_case': 'Total season exposure'
    },
    'season_average': {
        'method': 'mean',
        'description': 'Average weekly visibility score',
        'use_case': 'Consistency measurement'
    },
    'best_races': {
        'method': 'top_n_mean',
        'n': 10,
        'description': 'Average of top 10 race scores',
        'use_case': 'Peak performance measurement'
    }
}

# We'll use SEASON_TOTAL as primary metric
# Rationale: Sponsors care about total exposure, not just consistency

print("Aggregation Method: SEASON TOTAL")
print("="*50)
print("Each sponsor's weekly scores are summed to produce a season total.")
print("This reflects total visibility value delivered over the full season.")

Aggregation Method: SEASON TOTAL
Each sponsor's weekly scores are summed to produce a season total.
This reflects total visibility value delivered over the full season.


In [15]:
# Create sample test data
test_data = pd.DataFrame({
    'race_number': [1, 1, 2, 2, 3, 3],
    'sponsor': ['FedEx', 'NAPA', 'FedEx', 'NAPA', 'FedEx', 'NAPA'],
    'finish_position': [1, 15, 8, 3, 25, 2],
    'laps_led': [150, 0, 20, 80, 0, 100],
    'reddit_mentions': [45, 10, 15, 35, 5, 40],
    'youtube_sponsor_views': [500000, 50000, 100000, 300000, 25000, 400000],
    'news_weighted_mentions': [12, 3, 5, 10, 2, 11]
})

# Add binary flags
test_data['is_win'] = (test_data['finish_position'] == 1).astype(int)
test_data['is_playoff'] = (test_data['race_number'] >= 27).astype(int)

print("Test Data:")
print(test_data)

# Apply normalization
normalized = test_data.copy()
normalized['finish_norm'] = normalize_to_100(test_data['finish_position'], 'inverse')
normalized['laps_norm'] = normalize_to_100(test_data['laps_led'], 'direct')
normalized['reddit_norm'] = normalize_to_100(test_data['reddit_mentions'], 'direct')
normalized['youtube_norm'] = prepare_youtube_views(test_data['youtube_sponsor_views'])
normalized['news_norm'] = normalize_to_100(test_data['news_weighted_mentions'], 'direct')
normalized['win_norm'] = prepare_binary(test_data['is_win'])
normalized['playoff_norm'] = prepare_binary(test_data['is_playoff'])

print("\\nNormalized Values:")
norm_cols = ['sponsor', 'finish_norm', 'laps_norm', 'reddit_norm',
             'youtube_norm', 'news_norm', 'win_norm']
print(normalized[norm_cols].round(1))

Test Data:
   race_number sponsor  finish_position  laps_led  reddit_mentions  \
0            1   FedEx                1       150               45   
1            1    NAPA               15         0               10   
2            2   FedEx                8        20               15   
3            2    NAPA                3        80               35   
4            3   FedEx               25         0                5   
5            3    NAPA                2       100               40   

   youtube_sponsor_views  news_weighted_mentions  is_win  is_playoff  
0                 500000                      12       1           0  
1                  50000                       3       0           0  
2                 100000                       5       0           0  
3                 300000                      10       0           0  
4                  25000                       2       0           0  
5                 400000                      11       0           0  
\

In [16]:
def verify_scoring_formula(df, config, sample_index=0):
    """
    Step through scoring calculation for verification.
    Walk through every step of the calculation manually to confirm
    the stored score matches the calculated score.
    """
    row = df.iloc[sample_index]

    print("="*60)
    print(f"SCORING VERIFICATION: {row['sponsor']} at Race {row['race_number']}")
    print("="*60)

    # Show raw values
    print("\\n--- Raw Values ---")
    print(f"Finish Position: {row['finish_position']}")
    # ... more values

    # Show normalized values
    print("\\n--- Normalized Values ---")
    print(f"Finish Norm: {row['finish_norm']:.2f}")
    # ... more values

    # Calculate score manually
    manual_score = 0
    for var, weight in config['weights'].items():
        contribution = row[f'{var}_norm'] * weight
        manual_score += contribution
        print(f"{var}: {row[f'{var}_norm']:.2f} x {weight} = {contribution:.2f}")

    print(f"\\nCalculated Total: {manual_score:.2f}")
    print(f"Stored Score: {row['visibility_score']:.2f}")

    # The critical check
    if abs(manual_score - row['visibility_score']) < 0.01:
        print("\\n[OK] Score verification PASSED")
    else:
        print("\\n[ERROR] Score verification FAILED - check formula")

In [17]:
"""
NASCAR Sponsorship Visibility Scoring Model
==========================================

This notebook implements the visibility scoring methodology designed in Step 4.1.
It calculates weekly and season-total visibility scores for all tracked sponsors.

Author: Lloyd Todaro
Date: August 10th 2026
Version: 1.0
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Visibility Scoring Model - Initialization Complete")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Visibility Scoring Model - Initialization Complete
Timestamp: 2026-08-10 17:56


In [ ]:
# Load scoring configuration from Step 4.1
with open('scoring_config.json', 'r') as f:
    SCORING_CONFIG = json.load(f)

print("Scoring Configuration Loaded:")
print("="*50)
print(f"Version: {SCORING_CONFIG['version']}")
print(f"\\nCategory Weights:")
for cat, weight in SCORING_CONFIG['category_weights'].items():
    print(f"  {cat}: {weight*100:.0f}%")

# Load master dataset
df = pd.read_csv('data/processed/master_dataset.csv')

# Bring in the PCA-weighted visibility score computed once in
# data_visualization.ipynb (sponsor_summary['Visibility_Score_Shifted']),
# rather than re-deriving visibility from the raw Reddit/YouTube/News
# columns here with a different normalization scheme. Two independently
# built "visibility" numbers would disagree and defeat the point of having
# done the PCA weighting in the first place.
vis_scores = pd.read_csv('data/processed/visibility_scores.csv')
df = df.merge(vis_scores, on=['primary_sponsor', 'race_number'], how='left')

print(f"\\nDataset Loaded: {len(df)} rows x {len(df.columns)} columns")
print(f"Sponsors: {df['primary_sponsor'].unique().tolist()}")
print(f"Races: {df['race_number'].nunique()}")
print(f"Rows with visibility score: {df['Visibility_Score_Shifted'].notna().sum()}")

In [21]:
def normalize_min_max(series, direction='direct'):
    """
    Normalize a series to 0-100 scale using min-max normalization.

    Parameters:
    -----------
    series : pd.Series
        Values to normalize
    direction : str
        'direct' if higher is better, 'inverse' if lower is better

    Returns:
    --------
    pd.Series : Normalized values (0-100 scale)

    Examples:
    ---------
    >>> normalize_min_max(pd.Series([1, 5, 10]), 'direct')
    [0.0, 44.4, 100.0]
    >>> normalize_min_max(pd.Series([1, 5, 10]), 'inverse')
    [100.0, 55.6, 0.0]
    """
    min_val = series.min()
    max_val = series.max()

    # Handle edge case: all values identical
    if max_val == min_val:
        return pd.Series([50.0] * len(series), index=series.index)

    if direction == 'direct':
        normalized = (series - min_val) / (max_val - min_val) * 100
    elif direction == 'inverse':
        normalized = (max_val - series) / (max_val - min_val) * 100
    else:
        raise ValueError(f"Direction must be 'direct' or 'inverse', got {direction}")

    return normalized


def normalize_log_scale(series, direction='direct'):
    """
    Normalize a series using log transformation then min-max.
    Useful for highly skewed variables like YouTube views.

    Parameters:
    -----------
    series : pd.Series
        Values to normalize (must be non-negative)
    direction : str
        'direct' if higher is better

    Returns:
    --------
    pd.Series : Normalized values (0-100 scale)
    """
    # Log transform (add 1 to handle zeros)
    log_series = np.log1p(series)
    # Then apply standard min-max
    return normalize_min_max(log_series, direction)


def normalize_binary(series):
    """
    Scale binary (0/1) variable to 0-100.

    Parameters:
    -----------
    series : pd.Series
        Binary values (0 or 1)

    Returns:
    --------
    pd.Series : Scaled values (0 or 100)
    """
    return series * 100


# Test normalization functions
print("Testing Normalization Functions:")
print("="*50)

test_series = pd.Series([1, 5, 10, 20, 40])
print(f"Test data (positions): {test_series.tolist()}")
print(f"Direct normalization: {normalize_min_max(test_series, 'direct').round(1).tolist()}")
print(f"Inverse normalization: {normalize_min_max(test_series, 'inverse').round(1).tolist()}")

test_views = pd.Series([0, 1000, 10000, 100000, 1000000])
print(f"\\nTest data (views): {test_views.tolist()}")
print(f"Log normalization: {normalize_log_scale(test_views).round(1).tolist()}")

test_binary = pd.Series([0, 0, 1, 0, 1])
print(f"\\nTest data (binary): {test_binary.tolist()}")
print(f"Binary normalization: {normalize_binary(test_binary).tolist()}")

Testing Normalization Functions:
Test data (positions): [1, 5, 10, 20, 40]
Direct normalization: [0.0, 10.3, 23.1, 48.7, 100.0]
Inverse normalization: [100.0, 89.7, 76.9, 51.3, 0.0]
\nTest data (views): [0, 1000, 10000, 100000, 1000000]
Log normalization: [0.0, 50.0, 66.7, 83.3, 100.0]
\nTest data (binary): [0, 0, 1, 0, 1]
Binary normalization: [0, 0, 100, 0, 100]


In [ ]:
def prepare_scoring_data(df):
    """
    Prepare dataset for scoring by creating derived variables and normalizing.

    Parameters:
    -----------
    df : pd.DataFrame
        Master dataset with raw variables, already merged with the
        PCA-weighted Visibility_Score_Shifted column from data_visualization.ipynb

    Returns:
    --------
    pd.DataFrame : Dataset with normalized scoring variables
    """
    # Create a copy to avoid modifying original
    scoring_df = df.copy()

    # Create binary flags
    scoring_df['is_win'] = (scoring_df['finish_position'] == 1).astype(int)
    scoring_df['is_playoff'] = (
        (scoring_df['race_number'] >= 27) & (scoring_df['race_number'] < 37)
    ).astype(int)

    # Normalize each variable
    print("Normalizing variables...")

    # Race Performance
    scoring_df['finish_norm'] = normalize_min_max(
        scoring_df['finish_position'],
        direction='inverse'
    )
    scoring_df['laps_led_norm'] = normalize_min_max(
        scoring_df['laps_led'],
        direction='direct'
    )

    # Visibility: the PCA-weighted composite already combines Reddit,
    # YouTube, and News (see data_visualization.ipynb) — it does not get
    # rebuilt from those raw channels here. It only needs one final rescale
    # to the same 0-100 scale the other components use, since the PCA
    # loadings are unit-norm (sum of squares = 1) rather than a convex
    # combination (sum of absolute weights != 1), so the raw composite
    # isn't already on a comparable scale.
    scoring_df['visibility_norm'] = normalize_min_max(
        scoring_df['Visibility_Score_Shifted'],
        direction='direct'
    )

    # Special Events
    scoring_df['win_norm'] = normalize_binary(scoring_df['is_win'])
    scoring_df['playoff_norm'] = normalize_binary(scoring_df['is_playoff'])

    print("Normalization complete.")

    # Verify normalization ranges
    norm_cols = ['finish_norm', 'laps_led_norm', 'visibility_norm',
                 'win_norm', 'playoff_norm']

    print("\\nNormalized Variable Ranges:")
    for col in norm_cols:
        print(f"  {col}: {scoring_df[col].min():.1f} to {scoring_df[col].max():.1f}")

    return scoring_df


# Apply preparation
scoring_df = prepare_scoring_data(df)
print(f"\\nScoring dataset ready: {len(scoring_df)} rows")